In [ ]:
# ==============================
# 1. Import Libraries
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# ==============================
# 2. Load Dataset
# ==============================
# Check if Housing.csv exists, if not, download it from a public repository
import os
import requests

file_path = "/content/Housing.csv"
# Using a known public URL for the Housing.csv dataset
url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv" # Updated URL to a different source

if not os.path.exists(file_path):
    print(f"Downloading {os.path.basename(file_path)}...")
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status() # Raise an exception for HTTP errors
        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Download complete.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
        print("Please check the URL or your internet connection.")
        # Optionally, raise the exception or exit if the file is critical
else:
    print(f"{os.path.basename(file_path)} already exists.")

df = pd.read_csv(file_path)

# ==============================
# 3. Exploratory Data Analysis
# ==============================
print("First 5 rows:\n", df.head())
print("\nDataset Info:\n")
print(df.info())
print("\nStatistical Summary:\n", df.describe())

# ==============================
# 4. Check Missing Values
# ==============================
print("\nMissing Values:\n", df.isnull().sum())

# ==============================
# Fill missing values (for total_bedrooms) before further processing
df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)
print("\nMissing Values after imputation:\n", df.isnull().sum())

# ==============================
# 5. Outlier Analysis
# ==============================
fig, axs = plt.subplots(2, 3, figsize=(12, 6))

sns.boxplot(x=df['median_house_value'], ax=axs[0,0]) # Changed 'price' to 'median_house_value'
sns.boxplot(x=df['housing_median_age'], ax=axs[0,1]) # Changed 'area' to 'housing_median_age'
sns.boxplot(x=df['total_rooms'], ax=axs[0,2]) # Changed 'bedrooms' to 'total_rooms'
sns.boxplot(x=df['total_bedrooms'], ax=axs[1,0]) # Changed 'bathrooms' to 'total_bedrooms'
sns.boxplot(x=df['population'], ax=axs[1,1]) # Changed axs[1,1] to 'population'
sns.boxplot(x=df['households'], ax=axs[1,2]) # Changed axs[1,2] to 'households'

plt.tight_layout()
plt.show()

# ==============================
# 6. Pair Plot
# ==============================
sns.pairplot(df)
plt.show()

# ==============================
# 7. Data Preprocessing
# ==============================
# Features need to be adapted for the California Housing dataset
# Using some numerical features and 'ocean_proximity' for encoding
features = ['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'ocean_proximity']

df_encoded = pd.get_dummies(df[features], drop_first=True)

# Add target variable
df_encoded['median_house_value'] = df['median_house_value'] # Changed 'price' to 'median_house_value'

# ==============================
# 8. Train-Test Split
# ==============================
X = df_encoded.drop('median_house_value', axis=1) # Changed 'price' to 'median_house_value'
y = df_encoded['median_house_value'] # Changed 'price' to 'median_house_value'

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# 9. Build Model
# ==============================
model = LinearRegression()
model.fit(X_train, y_train)

# ==============================
# 10. Prediction
# ==============================
y_pred = model.predict(X_test)

# ==============================
# 11. Evaluation
# ==============================
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nModel Performance:")
print("Mean Squared Error:", mse)
print("R-squared:", r2)

# ==============================
# 12. Actual vs Predicted Plot
# ==============================
plt.figure(figsize=(8,5))
plt.scatter(y_test, y_pred, color='blue', alpha=0.5)

plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')

plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.title("Actual vs Predicted Prices")
plt.show()

# ==============================
# 13. Residual Plot
# ==============================
plt.figure(figsize=(8,5))
sns.residplot(x=y_pred, y=(y_test - y_pred), lowess=True, color='green')

plt.axhline(0, linestyle='--', color='red')
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot")

plt.show()